# 본선 — 시계열 신호 EDA 및 피처 엔지니어링

본선은 예선과 다른 데이터셋입니다. 각 trial 이 한 CSV 파일이고, 한 파일 안에 시계열 신호(`Time[s]`, `Signal A/B/C`, `Sensor A/B/C/D`) 가 들어 있으며 파일명에서 `weight` (라벨) 와 `driver` (A/B) 가 파싱됩니다.

이 노트북에서는 모든 train/test 파일을 읽어 다음을 수행합니다.

1. 신호 컬럼별 전역 min/max 를 한 번에 구하기
2. 전역 통계로 min-max 정규화
3. 파일 단위로 슬라이딩 윈도우(500 step) 슬라이스
4. 파일별 라벨/드라이버 부착
5. 학습/검증 split 후 parquet 저장


## Imports

In [ ]:
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed" / "finals"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

WINDOW_SIZE = 500
SPLIT_RATIO = 0.8
SIGNAL_COLUMNS = ["Time", "Signal A", "Signal B", "Signal C", "Sensor A", "Sensor B", "Sensor C", "Sensor D"]

## 파일 목록

In [ ]:
train_paths = sorted(glob.glob(str(DATA_DIR / "train" / "*.csv")))
test_paths = sorted(glob.glob(str(DATA_DIR / "test" / "*.csv")))
print(f"train files: {len(train_paths)}")
print(f"test files: {len(test_paths)}")

## 파일명 파서

In [ ]:
def parse_filename(path: str) -> tuple[float | None, int]:
    """Return (weight_label, driver_idx) parsed from filename like '<weight>kg_<driver>...'."""
    stem = Path(path).stem
    parts = stem.split("_")
    weight_label = None
    if parts and parts[0][:-2].replace(".", "").isdigit():
        weight_label = float(parts[0][:-2])
    driver_idx = 0 if (len(parts) > 1 and parts[1][:1] == "A") else 1
    return weight_label, driver_idx

## 단일 파일 시각화 (sanity check)

In [ ]:
sample_df = pd.read_csv(train_paths[len(train_paths) // 2]).rename(columns={"Time[s]": "Time"})
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for ax, channel in zip(axes, ["Signal A", "Signal B", "Sensor A"]):
    ax.plot(sample_df["Time"], sample_df[channel], color="steelblue", linewidth=0.7)
    ax.set_ylabel(channel)
axes[-1].set_xlabel("Time [s]")
plt.suptitle(f"sample trial — {Path(train_paths[len(train_paths) // 2]).name}")
plt.tight_layout()
plt.show()

## 전역 min/max 집계

In [ ]:
def collect_extremes(paths: list[str], columns: list[str]) -> dict[str, dict[str, float]]:
    extremes = {col: {"min": float("inf"), "max": float("-inf")} for col in columns}
    for path in tqdm(paths, desc="collect extremes"):
        data = pd.read_csv(path).rename(columns={"Time[s]": "Time"})
        for col in columns:
            extremes[col]["min"] = min(extremes[col]["min"], data[col].min())
            extremes[col]["max"] = max(extremes[col]["max"], data[col].max())
    return extremes


extremes = collect_extremes(train_paths + test_paths, SIGNAL_COLUMNS)
extremes_df = pd.DataFrame(extremes).T
extremes_df

## 윈도우 슬라이스 + 정규화

In [ ]:
def load_and_slice(path: str, extremes: dict, window: int, has_label: bool) -> list[pd.DataFrame]:
    data = pd.read_csv(path).rename(columns={"Time[s]": "Time"})
    label, driver_idx = parse_filename(path)
    data["driver"] = driver_idx
    if has_label:
        data["label"] = label
    for col, stats in extremes.items():
        spread = stats["max"] - stats["min"]
        if spread:
            data[col] = (data[col] - stats["min"]) / spread
    return [data.iloc[i:i + window] for i in range(0, len(data), window)]


def collect_chunks(paths: list[str], extremes: dict, has_label: bool) -> list[list[pd.DataFrame]]:
    return [load_and_slice(path, extremes, WINDOW_SIZE, has_label) for path in tqdm(paths)]


train_chunks = collect_chunks(train_paths, extremes, has_label=True)
test_chunks = collect_chunks(test_paths, extremes, has_label=False)
print(f"train chunks per file (sample 5): {[len(c) for c in train_chunks[:5]]}")

## 파일 균등 train/validation split

각 파일 안에서 셔플 후 8:2 로 분할합니다. 라벨 분포와 driver 분포가 한쪽에 쏠리지 않도록 파일 단위로 균등하게 split 합니다.


In [ ]:
rng = np.random.default_rng(SEED)
train_pieces, valid_pieces = [], []

for chunks in train_chunks:
    indices = np.arange(len(chunks))
    rng.shuffle(indices)
    cut = int(len(indices) * SPLIT_RATIO)
    train_pieces.extend([chunks[i] for i in indices[:cut]])
    valid_pieces.extend([chunks[i] for i in indices[cut:]])

X_train = pd.concat(train_pieces, ignore_index=True)
X_valid = pd.concat(valid_pieces, ignore_index=True)
test_chunks_flat = [chunk for chunks in test_chunks for chunk in chunks]
X_test = pd.concat(test_chunks_flat, ignore_index=True)

print(f"X_train: {X_train.shape}, X_valid: {X_valid.shape}, X_test: {X_test.shape}")

## 라벨 분포 확인

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(X_train["label"].drop_duplicates(), bins=30, color="steelblue", ax=ax)
ax.set_title("trial 단위 weight (label) 분포")
ax.set_xlabel("weight")
plt.tight_layout()
plt.show()

## 저장

In [ ]:
# 후속 노트북이 ID 기반 집계를 위해 test 의 ID 를 보존
test_ids = []
for path, chunks in zip(test_paths, test_chunks):
    file_id = Path(path).stem
    for chunk in chunks:
        test_ids.extend([file_id] * len(chunk))
X_test["ID"] = test_ids

X_train.to_parquet(PROCESSED_DIR / "X_train.parquet")
X_valid.to_parquet(PROCESSED_DIR / "X_valid.parquet")
X_test.to_parquet(PROCESSED_DIR / "X_test.parquet")
print("saved processed signal datasets to", PROCESSED_DIR)

## 정리

- 파일 단위 라벨/드라이버를 파일명에서 파싱했습니다.
- 신호 컬럼은 train+test 전체를 한 번에 스캔한 전역 min/max 로 정규화했습니다 (테스트가 학습 분포를 벗어나도 같은 스케일 유지).
- 각 파일을 500 step 슬라이딩 윈도우로 슬라이스하고 파일 단위로 8:2 split 했습니다.
- 결과는 parquet 으로 저장해 다음 노트북이 모델링/집계만 수행하도록 했습니다.
